# Remote Coding Agent: Colab + OpenAI Secure MCP Tunnel

This notebook starts the repository's real MCP server with MCP authentication disabled and unrestricted run_command, bound only to 127.0.0.1 inside Colab. It connects that private server to ChatGPT through OpenAI Secure MCP Tunnel. The tunnel client authenticates to OpenAI; the MCP server itself does not require a bearer token.

Important: Secure MCP Tunnel uses a tunnel_id, not a public server URL. Do not expose an unrestricted no-auth MCP server through a public Quick Tunnel. Before running, create/select a tunnel in OpenAI Platform, create a runtime API key with Tunnels Read + Use, and associate the tunnel with the ChatGPT workspace you will use.

The notebook checks the environment, installs/checks this repository's dependencies, verifies the current official tunnel-client download's SHA-256 digest, runs tunnel-client doctor --explain, checks local MCP initialization/tool discovery/a harmless command, then waits for a unique loopback health URL, /readyz, and observed successful OpenAI control-plane polling before reporting success.


In [ ]:
# Re-runnable deployment cell. No MCP bearer token is configured.
# The server stays on loopback; tunnel-client makes an outbound authenticated connection.
import getpass
import hashlib
import importlib.util
import json
import os
import pathlib
import platform
import re
import shutil
import signal
import subprocess
import sys
import tempfile
import time
import urllib.request
import zipfile

print("=== Preflight: machine and tools ===")
print("Python:", sys.version.split()[0])
print("Architecture:", platform.machine())
print("Git:", shutil.which("git") or "missing")
print("curl:", shutil.which("curl") or "missing")
print("Existing package imports:", {
    name: importlib.util.find_spec(name) is not None
    for name in ("mcp", "uvicorn", "httpx")
})

REPO_URL = "https://github.com/Dishant402955/remote-coding-agent.git"
REPO_DIR = pathlib.Path("/content/remote-coding-agent")
WORKSPACE = pathlib.Path("/content/remote-coding-agent-workspace")
SERVER_LOG = pathlib.Path("/content/remote-coding-agent-server.log")
TUNNEL_LOG = pathlib.Path("/content/remote-coding-agent-tunnel-client.log")
TUNNEL_BIN = pathlib.Path("/content/bin/tunnel-client")
MCP_LOCAL_URL = "http://127.0.0.1:8000/mcp"
previous_health_url_file = globals().get("HEALTH_URL_FILE")
if isinstance(previous_health_url_file, pathlib.Path):
    previous_health_url_file.unlink(missing_ok=True)
health_fd, health_url_path = tempfile.mkstemp(
    prefix="tunnel-client-health-", suffix=".url", dir="/content"
)
os.close(health_fd)
HEALTH_URL_FILE = pathlib.Path(health_url_path)
HEALTH_URL_FILE.unlink(missing_ok=True)

def stop_process(process):
    if process is None or process.poll() is not None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    deadline = time.monotonic() + 5
    while time.monotonic() < deadline:
        if process.poll() is not None:
            break
        time.sleep(0.1)
    if process.poll() is None:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        pass

# Stop processes left by a previous run of this cell.
stop_process(globals().get("tunnel_client"))
stop_process(globals().get("server"))

# Clone or update the public repository, then install this checkout.
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True, timeout=120,
    )
elif REPO_DIR.exists():
    raise RuntimeError(
        f"{REPO_DIR} exists but is not a Git repository. Inspect it before retrying."
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True, timeout=120,
    )

print("\n=== Installing/checking repository dependencies ===")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "httpx"],
    cwd=REPO_DIR, check=True, timeout=240,
)
for module in ("mcp", "uvicorn", "httpx", "remote_coding_agent"):
    if importlib.util.find_spec(module) is None:
        raise RuntimeError(f"Dependency check failed after installation: {module}")
if shutil.which("remote-coding-agent") is None:
    raise RuntimeError("The remote-coding-agent console command was not installed.")
print("Dependencies verified: mcp, uvicorn, httpx, remote_coding_agent, CLI entry point.")

WORKSPACE.mkdir(parents=True, exist_ok=True)

# Ask for tunnel details at runtime; never write credentials into notebook source.
tunnel_id = input("OpenAI Secure MCP Tunnel ID (tunnel_ + 32 lowercase hex): ").strip()
if not re.fullmatch(r"tunnel_[0-9a-f]{32}", tunnel_id):
    raise ValueError("Invalid tunnel ID format. Copy the ID from OpenAI Platform → Tunnels.")
control_plane_api_key = getpass.getpass(
    "OpenAI runtime API key (input is hidden; do not paste an admin key): "
).strip()
if not control_plane_api_key:
    raise ValueError("A runtime API key is required by tunnel-client.")
print("Tunnel ID format: valid. Runtime key entered without echo.")

# Download the latest official Linux tunnel-client release and verify its SHA-256.
machine = platform.machine().lower()
asset_arch = {
    "x86_64": "amd64",
    "amd64": "amd64",
    "aarch64": "arm64",
    "arm64": "arm64",
}.get(machine)
if asset_arch is None:
    raise RuntimeError(f"Unsupported CPU architecture for tunnel-client: {machine}")

request = urllib.request.Request(
    "https://api.github.com/repos/openai/tunnel-client/releases/latest",
    headers={"Accept": "application/vnd.github+json"},
)
with urllib.request.urlopen(request, timeout=30) as response:
    release = json.load(response)
tag = release.get("tag_name")
asset_name = f"tunnel-client-{tag}-linux-{asset_arch}.zip"
asset = next((item for item in release.get("assets", []) if item.get("name") == asset_name), None)
if not tag or asset is None or not str(asset.get("digest", "")).startswith("sha256:"):
    raise RuntimeError(f"Could not find a verifiable official release asset: {asset_name}")

archive_path = pathlib.Path("/content/tunnel-client-release.zip")
digest = hashlib.sha256()
downloaded = 0
with urllib.request.urlopen(asset["browser_download_url"], timeout=60) as response:
    with archive_path.open("wb") as target:
        while chunk := response.read(1024 * 1024):
            downloaded += len(chunk)
            if downloaded > 100 * 1024 * 1024:
                raise RuntimeError("tunnel-client archive exceeded the 100 MiB safety limit.")
            digest.update(chunk)
            target.write(chunk)
if f"sha256:{digest.hexdigest()}" != asset["digest"]:
    archive_path.unlink(missing_ok=True)
    raise RuntimeError("tunnel-client SHA-256 check failed; refusing to execute the download.")

extract_dir = pathlib.Path("/content/tunnel-client-release")
shutil.rmtree(extract_dir, ignore_errors=True)
extract_dir.mkdir(parents=True)
with zipfile.ZipFile(archive_path) as archive:
    for item in archive.infolist():
        destination = (extract_dir / item.filename).resolve()
        is_inside_extract_dir = extract_dir.resolve() in destination.parents
        is_extract_dir = destination == extract_dir.resolve()
        if not is_inside_extract_dir and not is_extract_dir:
            raise RuntimeError("Unsafe path found in tunnel-client release archive.")
    archive.extractall(extract_dir)

candidates = [path for path in extract_dir.rglob("tunnel-client") if path.is_file()]
if not candidates:
    raise RuntimeError("Verified release archive did not contain the tunnel-client binary.")
TUNNEL_BIN.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(candidates[0], TUNNEL_BIN)
TUNNEL_BIN.chmod(0o755)
version_check = subprocess.run(
    [str(TUNNEL_BIN), "help", "quickstart"],
    text=True, capture_output=True, timeout=20,
)
if version_check.returncode != 0:
    raise RuntimeError(
        "Downloaded tunnel-client did not pass its quickstart check: "
        + (version_check.stderr or version_check.stdout)[-2000:]
    )
print(f"Official tunnel-client installed and verified: {tag} ({asset_arch}).")

# Start the MCP server without its own authentication, but only on loopback.
server_env = os.environ.copy()
for key in list(server_env):
    if any(word in key.upper() for word in (
        "TOKEN", "SECRET", "PASSWORD", "API_KEY", "ADMIN_KEY",
        "PRIVATE_KEY", "AUTHORIZATION", "CREDENTIAL",
    )):
        server_env.pop(key, None)
server_env.update({
    "REMOTE_CODING_AGENT_AUTH_MODE": "none",
    "REMOTE_CODING_AGENT_TEST_MODE": "false",
    "REMOTE_CODING_AGENT_HOST": "127.0.0.1",
    "REMOTE_CODING_AGENT_PORT": "8000",
    "REMOTE_CODING_AGENT_WORKSPACE": str(WORKSPACE),
})
server_log_handle = SERVER_LOG.open("w", encoding="utf-8")
server = subprocess.Popen(
    ["remote-coding-agent"],
    cwd=REPO_DIR,
    env=server_env,
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

httpx = importlib.import_module("httpx")
ClientSession = importlib.import_module("mcp").ClientSession
streamablehttp_client = importlib.import_module(
    "mcp.client.streamable_http"
).streamablehttp_client

deadline = time.monotonic() + 30
while time.monotonic() < deadline:
    if server.poll() is not None:
        server_log_handle.close()
        raise RuntimeError(
            "MCP server exited during startup. Last log lines:\n"
            + "\n".join(SERVER_LOG.read_text(errors="replace").splitlines()[-30:])
        )
    try:
        response = httpx.get("http://127.0.0.1:8000/", timeout=1)
        if response.status_code in (200, 401, 404, 405):
            break
    except httpx.HTTPError:
        time.sleep(0.25)
else:
    stop_process(server)
    server_log_handle.close()
    raise RuntimeError("MCP server did not respond on loopback within 30 seconds.")

async def verify_local_mcp():
    async with streamablehttp_client(MCP_LOCAL_URL) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await session.list_tools()
            names = [tool.name for tool in tools.tools]
            if "run_command" not in names:
                raise RuntimeError(f"run_command not advertised; discovered tools: {names}")
            result = await session.call_tool(
                "run_command", {"command": "printf 'local-mcp-ok'"},
            )
            if result.isError:
                raise RuntimeError("Local MCP tool call returned an MCP error.")
            structured = result.structuredContent
            if structured is None:
                for item in result.content:
                    text_value = getattr(item, "text", None)
                    if text_value:
                        try:
                            structured = json.loads(text_value)
                            break
                        except json.JSONDecodeError:
                            pass
            if not isinstance(structured, dict):
                raise RuntimeError("Local MCP tool call returned no structured result.")
            if structured.get("exit_code") != 0 or structured.get("stdout") != "local-mcp-ok":
                raise RuntimeError(f"Local MCP smoke test failed: {structured}")
            return names, structured

# Use top-level await in the notebook kernel for the local MCP verification.
try:
    local_tool_names, local_tool_result = await verify_local_mcp()
except Exception:
    stop_process(server)
    server_log_handle.close()
    raise
print("Local MCP initialize/list_tools/run_command: PASS")
print("Discovered tools:", local_tool_names)
print("Safe command output:", local_tool_result["stdout"])

# Start OpenAI's outbound-only tunnel client. MCP auth stays disabled; the client
# authenticates to the OpenAI tunnel service using the runtime key.
tunnel_env = os.environ.copy()
for key in list(tunnel_env):
    if any(word in key.upper() for word in (
        "TOKEN", "SECRET", "PASSWORD", "API_KEY", "ADMIN_KEY",
        "PRIVATE_KEY", "AUTHORIZATION", "CREDENTIAL",
    )):
        tunnel_env.pop(key, None)
tunnel_env.update({
    "CONTROL_PLANE_API_KEY": control_plane_api_key,
    "CONTROL_PLANE_TUNNEL_ID": tunnel_id,
    "MCP_SERVER_URL": MCP_LOCAL_URL,
    "HEALTH_LISTEN_ADDR": "127.0.0.1:0",
    "HEALTH_URL_FILE": str(HEALTH_URL_FILE),
})
HEALTH_URL_FILE.unlink(missing_ok=True)

# Run the official tunnel-client preflight against the private MCP endpoint.
doctor_result = subprocess.run(
    [str(TUNNEL_BIN), "doctor", "--explain"],
    cwd=REPO_DIR,
    env=tunnel_env,
    text=True,
    capture_output=True,
    timeout=45,
)
doctor_output = doctor_result.stdout + "\n" + doctor_result.stderr
doctor_output = doctor_output.replace(control_plane_api_key, "<redacted>")
if doctor_result.returncode != 0:
    stop_process(server)
    server_log_handle.close()
    raise RuntimeError(
        "tunnel-client doctor --explain failed; resolve the preflight diagnostics "
        "before starting the tunnel client:\n" + doctor_output[-5000:]
    )
print("tunnel-client doctor --explain: PASS")

tunnel_log_handle = TUNNEL_LOG.open("w", encoding="utf-8")
tunnel_client = subprocess.Popen(
    [str(TUNNEL_BIN), "run", "--log.level=info", "--log.format=struct-text"],
    cwd=REPO_DIR,
    env=tunnel_env,
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

ready = False
health_base_url = None
control_plane_health = None
deadline = time.monotonic() + 90
while time.monotonic() < deadline:
    if tunnel_client.poll() is not None:
        break
    try:
        if HEALTH_URL_FILE.is_file():
            candidate = HEALTH_URL_FILE.read_text(encoding="utf-8").strip().rstrip("/")
            # Health endpoints must remain loopback-only, even if config is changed.
            if candidate.startswith("http://127.0.0.1:"):
                health_base_url = candidate
                health = httpx.get(f"{health_base_url}/healthz", timeout=2)
                readiness = httpx.get(f"{health_base_url}/readyz", timeout=2)
                if health.status_code == 200 and readiness.status_code == 200:
                    poll_response = httpx.get(
                        f"{health_base_url}/health/control-plane", timeout=2
                    )
                    if poll_response.status_code == 200:
                        control_plane_health = poll_response.json()
                        poll_details = control_plane_health.get("details") or {}
                        # Ready is not enough: require observed success polling OpenAI.
                        if (
                            control_plane_health.get("status") == "ok"
                            and poll_details.get("last_success")
                            and poll_details.get("consecutive_failures", 0) == 0
                        ):
                            ready = True
                            break
    except (OSError, httpx.HTTPError, ValueError):
        pass
    time.sleep(1)

tunnel_log_handle.close()
server_log_handle.close()
if not ready:
    stop_process(tunnel_client)
    stop_process(server)
    tail = "\n".join(TUNNEL_LOG.read_text(errors="replace").splitlines()[-40:])
    raise RuntimeError(
        "Secure MCP Tunnel did not become ready within 90 seconds. "
        "Check the tunnel ID, runtime key permissions (Tunnels Read + Use), "
        "and that this tunnel is associated with the ChatGPT workspace. "
        "Recent tunnel-client logs (redacted):\n"
        + tail.replace(control_plane_api_key, "<redacted>")
    )

print("\n=== Deployment verified ===")
print("MCP server: running on loopback only")
print("MCP authentication: disabled")
print("Command allowlist: disabled; run_command executes commands permitted to this Colab runtime")
print("Local MCP endpoint (not a public URL):", MCP_LOCAL_URL)
print("OpenAI Secure MCP Tunnel ID:", tunnel_id)
print("tunnel-client /healthz: PASS")
print("tunnel-client /readyz: PASS")
print("OpenAI control-plane polling: PASS (successful poll observed)")
print("Tunnel-client health base URL (loopback only):", health_base_url)
print("\n=== Create the ChatGPT connection ===")
print("1. In ChatGPT web, open Plugins → + → Add custom MCP server.")
print("2. Set Connection to Tunnel, then select this tunnel or enter the ID printed above.")
print("3. Set MCP authentication to None if the form asks; the server itself has no bearer auth.")
print("4. Review the risk warning and create the plugin.")
print("5. Keep this Colab runtime and this notebook session running while using the MCP server.")
print("No public HTTPS URL is created; Secure MCP Tunnel uses the tunnel ID.")


## Stop the server and tunnel client

Run this cell when finished. It stops both processes, clears the runtime API key from the current notebook variables, and removes the ephemeral health URL file.

```python
stop_process(globals().get("tunnel_client"))
stop_process(globals().get("server"))
if isinstance(globals().get("tunnel_env"), dict):
    tunnel_env.pop("CONTROL_PLANE_API_KEY", None)
health_url_file = globals().get("HEALTH_URL_FILE")
if isinstance(health_url_file, pathlib.Path):
    health_url_file.unlink(missing_ok=True)
tunnel_env = None
control_plane_api_key = None
print("MCP server and Secure MCP Tunnel client stopped.")
```

The server has no MCP-layer authentication, but it listens only on 127.0.0.1; OpenAI's Secure MCP Tunnel is the remote access path. To reconnect after a runtime reset, rerun the deployment cell and enter the tunnel ID and runtime key again.